<a href="https://colab.research.google.com/github/abhayjr11/goalkeep_assessment/blob/main/goalkeep_sql_ex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
from google.colab import drive
drive.mount('/content/drive')
import os
# dataset file path
path = "/content/drive/MyDrive/pyspark/goalkeep/NYC_Service_Requests.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [113]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
# Read CSV
df = spark.read.csv(path, header=True)

df1 = df.withColumn(
    "created_date",
    to_timestamp(col("created_date"), "MM/dd/yyyy h:mm:ss a")
)

df1.write.mode("overwrite").saveAsTable("service_requests")

In [114]:
query = '''SELECT
sr.zip_code,
sr.agency,
COUNT(*) AS num_complaints
FROM (
SELECT *
FROM service_requests

WHERE complaint_type = 'Noise - Residential'
) AS sr
JOIN (
SELECT *
FROM service_requests
WHERE created_date >= CURRENT_DATE - INTERVAL '390 days'
) AS recent
ON sr.unique_key = recent.unique_key
JOIN (
SELECT DISTINCT zip_code
FROM service_requests
WHERE zip_code IS NOT NULL
) AS zip_filter
ON sr.zip_code = zip_filter.zip_code
WHERE sr.closed_date IS NOT NULL
AND sr.zip_code IS NOT NULL
GROUP BY sr.zip_code, sr.agency
ORDER BY num_complaints DESC
LIMIT 10;'''

spark.sql(query).show()

+--------+------+--------------+
|zip_code|agency|num_complaints|
+--------+------+--------------+
|   10457|  NYPD|            28|
|   11207|  NYPD|            23|
|   11221|  NYPD|            23|
|   10468|  NYPD|            20|
|   10462|  NYPD|            19|
|   11226|  NYPD|            18|
|   11249|  NYPD|            18|
|   11385|  NYPD|            18|
|   10467|  NYPD|            18|
|   10463|  NYPD|            17|
+--------+------+--------------+



In [117]:
qr = '''select distinct zip_code, agency, count(*)  as num_complaints from service_requests
WHERE complaint_type = 'Noise - Residential'
and created_date >= CURRENT_DATE - INTERVAL '400 days'
and closed_date IS NOT NULL
and zip_code IS NOT NULL
group by 1,2
order by num_complaints  DESC
limit 10;'''

spark.sql(qr).show()

+--------+------+--------------+
|zip_code|agency|num_complaints|
+--------+------+--------------+
|   10457|  NYPD|            28|
|   11207|  NYPD|            23|
|   11221|  NYPD|            23|
|   10468|  NYPD|            20|
|   10462|  NYPD|            19|
|   11226|  NYPD|            18|
|   11249|  NYPD|            18|
|   11385|  NYPD|            18|
|   10467|  NYPD|            18|
|   10463|  NYPD|            17|
+--------+------+--------------+



In [120]:
q = '''with cte_com_type as (
select zip_code, agency,  created_date, closed_date
from service_requests
WHERE complaint_type = 'Noise - Residential'

),

cte_within_time_interval as (
select zip_code, agency, created_date, closed_date
from cte_com_type
where created_date >= CURRENT_DATE - INTERVAL '400 days'
)

select zip_code, agency, count(*) as num_complaints
from cte_within_time_interval
where closed_date IS NOT NULL and zip_code IS NOT NULL
group by zip_code, agency
order by num_complaints DESC
limit 10;'''

spark.sql(q).show()

+--------+------+--------------+
|zip_code|agency|num_complaints|
+--------+------+--------------+
|   10457|  NYPD|            28|
|   11207|  NYPD|            23|
|   11221|  NYPD|            23|
|   10468|  NYPD|            20|
|   10462|  NYPD|            19|
|   11226|  NYPD|            18|
|   11249|  NYPD|            18|
|   11385|  NYPD|            18|
|   10467|  NYPD|            18|
|   10463|  NYPD|            17|
+--------+------+--------------+

